# Notebook 2: Fine-tune CHURRO and score it

This trains a **LoRA adapter** for **CHURRO-3B**, a handwriting model trained on 100,000
historical pages, on whole pages from all three collections at once, and scores the result on the
team's 55 dev pages with the competition's own `metric.py`, per collection.

The training is tailored to the problems Lokesh's dev-set audit found. Each problem has a fix, and
each fix is a setting you can turn off to measure what it's worth (Step 4 lists them).

**The notebook instance can be a small `ml.t3.medium`.** With `WHERE = "job"` (Step 4), each run
goes to a **SageMaker training job** on an `ml.g5.xlarge` GPU, which is billed only while that run
goes; this notebook just starts it and shows its progress. With `WHERE = "here"` the run uses this
instance's own GPU instead (an `ml.g5.xlarge` notebook, billed every hour it is switched on; on a
T4 `ml.g4dn` it is two to three times slower, and fp16 training there can fail with a `nan` loss).
Run the cells from top to bottom with **Shift + Enter**. Kernel: **conda_pytorch_p310** (or another `conda_pytorch` kernel, not `conda_python3`).

## Step 1. Install the libraries

Run this after every start of the instance, and before any other cell.

In [ ]:
%pip install -q -r requirements.txt

import importlib.metadata
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
    raise SystemExit("This kernel has no PyTorch, so it can't run the model. Choose Kernel → Change Kernel, "
                     "pick conda_pytorch_p310 (or another conda_pytorch kernel), then run this cell again.")

# The model also needs torchvision in exactly the version and build made for this kernel's
# PyTorch (2.8.0 goes with 0.23.0, 2.9.1 with 0.24.1, ...). conda_pytorch kernels have it.
def torchvision_works():
    return subprocess.run([sys.executable, "-c", "import torchvision"], capture_output=True).returncode == 0

torch_version, _, build = importlib.metadata.version("torch").partition("+")
if not torchvision_works():
    _, minor, patch = (torch_version.split(".") + ["0"])[:3]
    wanted = f"torchvision==0.{int(minor) + 15}.{patch}"
    sources = ([["--index-url", f"https://download.pytorch.org/whl/{build}"]] if build else []) + [[]]
    for source in sources:
        print(f"Installing {wanted} for PyTorch {torch_version}{'+' + build if build else ''} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", wanted, *source])
        if torchvision_works():
            break
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchvision"])
    else:
        raise SystemExit("Couldn't install a torchvision that works with this kernel's PyTorch. Switch the "
                         "kernel to conda_pytorch_p310 (Kernel → Change Kernel) and run this cell again.")
print(f"PyTorch {importlib.metadata.version('torch')}, torchvision {importlib.metadata.version('torchvision')}: ready.")

## Step 2. Set up the folders

In [ ]:
import importlib.util
import os
import subprocess
import sys

# Which kernel is this? It should be conda_pytorch_p310 (or another conda_pytorch kernel).
print("Kernel environment:", os.path.basename(os.path.dirname(os.path.dirname(sys.executable))))
missing = [pkg for module, pkg in (("PIL", "pillow"), ("pandas", "pandas"))
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installing", ", ".join(missing), "into this kernel ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# Everything this project downloads or creates lives here, on the notebook
# instance's storage volume, so it survives Stop and Start.
WORK = "/home/ec2-user/SageMaker/churro-work"
DATA = os.path.join(WORK, "data")     # page images and lists of pages
RUNS = os.path.join(WORK, "runs")     # one folder per training run
SCRIPTS = os.path.abspath("scripts")  # the scripts folder next to this notebook
for folder in (WORK, DATA, RUNS):
    os.makedirs(folder, exist_ok=True)
sys.path.insert(0, SCRIPTS)
os.environ["HF_HOME"] = os.path.join(WORK, "hf_cache")   # the 7.5 GB model download goes here

# Your bucket name is saved here, outside the repository, so it's typed once and
# updating the code (git pull) never touches it.
import json
SETTINGS = os.path.join(WORK, "my_settings.json")
def saved_setting(name, default=""):
    try:
        return json.load(open(SETTINGS)).get(name, default)
    except (OSError, ValueError):
        return default
def save_setting(name, value):
    try:
        current = json.load(open(SETTINGS))
    except (OSError, ValueError):
        current = {}
    current[name] = value
    json.dump(current, open(SETTINGS, "w"), indent=2)
print("Working folder:", WORK)

## Step 3. Check the GPU, the disk and the data

In [ ]:
import shutil
import torch

if not torch.cuda.is_available():
    print('No GPU on this instance, so runs go to a SageMaker training job (Step 4, WHERE = "job").')
else:
    gpu = torch.cuda.get_device_properties(0)
    bf16 = torch.cuda.get_device_capability()[0] >= 8
    print(f"GPU:  {gpu.name}, {gpu.total_memory / 1e9:.0f} GB, {'bf16' if bf16 else 'fp16 (no bf16 on this GPU)'}")
    if not bf16:
        print("WARNING: this GPU (a T4?) trains in fp16, which can give a nan loss with this model, and is 2-3x slower.\n"
              "  An ml.g5.xlarge or ml.g6.xlarge instance is the safer choice (README.md, Part 1).")
    # The model is about 7.5 GB, downloaded once into churro-work/hf_cache. Check there's room first.
    import glob
    free = shutil.disk_usage(WORK).free / 1e9
    downloaded = sum(os.path.getsize(f) for f in glob.glob(os.path.join(WORK, "hf_cache", "hub", "models--stanford-oval--churro-3B",
                                                                         "blobs", "*")) if not f.endswith(".incomplete")) / 1e9
    needed = max(0.0, 7.5 - downloaded) + 3   # what's left of the model, plus room for a run's checkpoints
    print(f"Disk: {free:.1f} GB free in ~/SageMaker; the model needs {max(0.0, 7.5 - downloaded):.1f} GB more")
    if free < needed:
        biggest = subprocess.run("du -sh ~/SageMaker/* ~/SageMaker/*/* 2>/dev/null | sort -h | tail -8",
                                 shell=True, capture_output=True, text=True).stdout
        raise SystemExit(f"Not enough disk space: {free:.1f} GB free, about {needed:.0f} GB needed.\n"
                         "Either make the volume bigger (Stop the instance, Edit, Volume size 50 GB, Start; your files are kept), "
                         "or delete something you don't need, e.g. ~/SageMaker/badger-finetune/hf_cache (the Kurrent pipeline's "
                         "9 GB model copy, downloaded again if that pipeline runs). The biggest folders:\n" + biggest)
missing = [n for n in ("train_items.csv", "dev_pages.csv", "parts.csv") if not os.path.exists(os.path.join(DATA, n))]
if missing:
    raise SystemExit(f"Missing {', '.join(missing)}. Run Notebook 1 first.")
import pandas as pd
items = pd.read_csv(os.path.join(DATA, "train_items.csv"), keep_default_na=False)
print("Training images:", ", ".join(f"{c} {n}" for c, n in items.category.value_counts().items()))

## Step 4. Choose the run and the settings

| `RUN` | What it does | About how long on `ml.g5.xlarge` |
|---|---|---|
| `"smoke"` | 12 images, 1 epoch, 3 dev pages. Only proves everything works. | 10–15 min the first time (mostly downloading the 7.5 GB model, once) |
| `"plain"` | No training and none of the reading fixes below, on all 55 dev pages. (Survey spreads are still read one page at a time; that's set in Notebook 1.) | 15–30 min |
| `"baseline"` | No training, but with the reading fixes below. The difference from `"plain"` is what the fixes are worth on their own. | 15–30 min |
| `"full"` | Trains with all the settings below, then scores all 55 dev pages. | 1 to 1.5 hours |
| `"kade-baseline"` | No training. Scores the held-out Kade documents of split `SPLIT`. The number each Kade run must beat. | 10–20 min |
| `"kade"` | Trains on Kade only, without the held-out documents, and scores those documents after **every** epoch, keeping each epoch's adapter. | 2–3 hours for 5 epochs |
| `"survey-baseline"` | No training. Scores the 35 survey dev pages. The number each survey run must beat. | 15–25 min |
| `"survey"` | Trains on the survey notes only, and scores all 35 survey dev pages after every epoch, keeping each epoch's adapter. | 2–3 hours for 3 epochs |
| `"rescore"` | No training. Scores a saved adapter (`ADAPTER`, e.g. a Kade run's `epoch3` folder) on the same pages its run scored, with the reading settings below. Tests a reading change on a trained model in one go. | 15–30 min |
| `"kade-more"` | Trains **on from** a saved Kade adapter (`INIT_ADAPTER`, e.g. a `kadeA` run's `epoch3`) with a gentle learning rate, now adding kade_007, and holds out only kade_112. | about 1 hour for 2 epochs |
| `"kade-survey"` | Trains on from a saved Kade adapter (`INIT_ADAPTER`) on Kade **and** survey notes together, holding out kade_112 and the survey dev pages, and scores 10 pages of each after every epoch. | 2–3 hours for 2 epochs |
| `"dominy"` | Trains on the Dominy pages only, each used 4 times per epoch, with the contrast stretch on, and scores all 9 Dominy dev pages after every epoch for 4 epochs, keeping each epoch's adapter. Starts from `INIT_ADAPTER` if it is set (lr 1e-4), or from plain CHURRO if it is `""` (lr 2e-4). | about 1 hour |
| `"dominy-final"` | The same, trained on every Dominy page, dev pages included, for `FINAL_EPOCHS`. | under 1 hour |
| `"kade-final"`, `"survey-final"` | Trains one collection on **every** training page, dev pages included, for `FINAL_EPOCHS`. The model for the test pages. Its dev score means nothing, since it trained on those pages. | 1–3 hours |

**The survey runs** use the usual 35 survey dev pages. Every survey page comes from the same
1853 notebook, and Kaggle picked its hidden test pages the same way, so here (unlike Kade) the dev
pages are a fair yardstick.

**Training on from an adapter.** `"kade-more"` and `"kade-survey"` start from what a saved adapter
already learned, with a new learning-rate schedule. (Running a finished run again doesn't do this:
it's refused, so its results aren't overwritten.) kade_112 is only 11 small postcards, so their dev
score is noisy; judge by the page-by-page comparison.

**More reading fixes to try with `"rescore"`**, all off by default: `no_repeat_ngram_size` (e.g. 12)
never writes the same run of words twice; `retry_repetition_penalty` (e.g. 1.15) reads a part that
looked wrong more strictly the second time; `compare_adapter` also reads every page with a second
adapter (e.g. another epoch's) and keeps that reading when the main one is over 1.5 times longer,
which catches runaways on short pages.

**Two reading fixes to try with `"rescore"`**, both off by default so you can measure each:
`cut_repeats` cuts a reading where it starts repeating itself (untrained CHURRO often reads a
letter, then keeps inventing text), and `retry_pick = "closer"` keeps the second reading of a part
only when its length is closer to normal.

**Why the Kade runs exist.** All 11 Kade dev pages come from one document (kade_112), so the
usual dev score can't say how a new document will read, or whether training has gone on too long.
The Kade runs leave out whole documents instead, and report each one separately, with and without
the postmark line (it's on the back of the card, so no model can read it). Split `"A"` trains on
kade_103 + kade_128 and scores kade_007 + kade_112. Split `"B"` trains on kade_103 + kade_007 and
scores kade_128 + kade_112. Pick settings on split A, then confirm the winner on split B.

On an `ml.g4dn.xlarge`, multiply by about 2.5.

**How each problem in the dev-set audit is tackled.** The numbers are the audit's own problem
numbers.

| Audit problem | What this does about it | Setting |
|---|---|---|
| 1. Survey spreads read in the wrong order, stopped early, or read twice | Spreads are cut at the fold and read one page at a time (Notebook 1). A reading that comes out empty, far too short, or stuck repeating itself is read again in two parts, top and bottom. | `retry` |
| 2. Kade dev postcards are small, faint pencil | Small images are enlarged to the same detail as the others. In training, Kade images are sometimes shrunk and faded, so the model also learns from pages like the postcards. | `upscale`, `augment` |
| 3. Kade postmark line isn't on the page | Never trained on. On small cards (68% of the small training cards have one) the part every postmark line shares, `Postmark , 19`, is added: on average it saves more than it costs. | `postmark_guess` |
| 4. Dialect spelling (*jestern*, *bei die factry*) | No spelling correction anywhere; labels are learned exactly as written. | (always) |
| 5. Stained, bleed-through Dominy pages; `#` in labels | `#` is never something the model learns to write. Training images sometimes get a faint mirror image of themselves, like ink from the back. An optional contrast stretch. | `augment`, `enhance` |
| 6. Ledger symbols the labels leave out | Check marks and `×` are removed from the output; curly quotes and long dashes become plain ones. | (always) |
| 7. Few pages, short pages, and every collection a third of the score | Dominy's 27 training pages are used 4 times per epoch, each time with different random changes. | `oversample` |
| 8. Survey vocabulary and numbers | Learned from 365 survey training images. | (training) |
| 9. Labels that are machine drafts | Scores on the expert-typed dev pages are shown separately. | (scoring) |

README.md explains every setting in plain words. **Change one at a time**, and write each run's
settings and scores in the team's results table. Each combination gets its own folder in `runs/`;
running it again continues from its last saved epoch, and deleting its folder starts it over.

In [ ]:
WHERE = "job"   # "job": a SageMaker training job rents the GPU for this run only (this notebook can be a
                #   small ml.t3.medium). "here": this notebook instance's own GPU.
JOB_INSTANCE = "ml.g5.xlarge"   # the job's GPU: one A10G with 24 GB, the same as the g5 notebook's
JOB_HOURS = 6                   # the job is stopped after this many hours, whatever happens: a cost cap
SPOT = False                    # True: spare capacity, much cheaper, but it can be interrupted (it then carries
                                #   on from its last checkpoint). Needs a spot quota for training jobs.

RUN = "survey"  # "smoke", "plain", "baseline", "full"; per collection: "kade-baseline", "kade", "survey-baseline", "survey";
                # "dominy"; then "rescore" to test reading settings on a saved adapter, and "kade-final" / "survey-final" / "dominy-final"
SPLIT = "A"     # Kade runs only: which documents are held out (see the table above)
HOLDOUT = {"A": "kade_007,kade_112", "B": "kade_128,kade_112"}
ADAPTER = ""    # "rescore" only: a saved adapter folder, e.g. os.path.join(RUNS, "<a kade run>", "epoch3")
FINAL_EPOCHS = 3   # "kade-final" / "survey-final" only: the best epoch from that collection's runs
INIT_ADAPTER = ""  # "kade-more" / "kade-survey" (required), "dominy" / "dominy-final" (optional; "" = plain CHURRO):
                   # e.g. os.path.join(RUNS, "kadeA-ep5-lr0.0002-r16-augall-vis0-px1003520-up1-enhnone-re1-pm1-rp1.05", "epoch3")

settings = {
    # training
    "epochs": 3,                         # passes through all the training images
    "lr": 2e-4,                          # learning rate: how big each correction is
    "lora_r": 16,                        # adapter size
    "lora_alpha": 32,                    # adapter strength; keep at 2 x lora_r
    "grad_accum": 8,                     # images per correction
    "augment": "all",                    # random changes to training images: "all", "none", or e.g. "dominy_accounts"
    "oversample": "dominy_accounts:4",   # use a collection's images several times per epoch ("" = none)
    "train_vision": 0,                   # 1 = also train the image side of the model
    # reading (training and scoring alike)
    "max_pixels": 1280 * 28 * 28,        # image detail: about a megapixel
    "upscale": 1,                        # 1 = enlarge small images (the Kade postcards)
    "enhance": "",                       # contrast stretch for these collections, e.g. "dominy_accounts"
    "retry": 1,                          # 1 = read a part again if its reading looks wrong
    "postmark_guess": 1,                 # 1 = add "Postmark , 19" on small Kade cards
    "repetition_penalty": 1.05,          # above 1.0 discourages repeating text; 1.0 = off
    "max_new_tokens": 2500,              # room for long ledger pages; the default 1536 may cut them off
    "retry_pick": "longer",              # "closer" = keep the second reading only if its length is closer to normal
    "cut_repeats": "",                   # cut readings where they start repeating, for these collections, e.g. "kade_letters"
    "no_repeat_ngram_size": 0,           # never write the same run of this many tokens twice (0 = off; try 12)
    "retry_repetition_penalty": 0,       # stricter repetition penalty for a second reading (0 = same; try 1.15)
    "compare_adapter": "",               # "rescore" only: a second adapter folder; its reading is kept when the main one runs away
    # scoring
    "curve_pages": 3,                    # dev pages per collection scored after every epoch
}
READING = ("max_pixels", "upscale", "enhance", "retry", "postmark_guess", "repetition_penalty", "retry_pick", "cut_repeats",
           "no_repeat_ngram_size", "retry_repetition_penalty", "compare_adapter")
reading = {k: settings[k] for k in READING}
kade = {"categories": "kade_letters", "holdout_docs": HOLDOUT[SPLIT]}
survey = {"categories": "survey_notes"}
solo = dict(oversample="", curve_pages=99, save_every_epoch=1, train_score_pages=10)   # one collection's runs
if RUN == "smoke":
    settings.update(epochs=1, max_train_items=12, dev_pages=3, curve_pages=0)
elif RUN == "plain":
    settings = {"epochs": 0, "max_pixels": settings["max_pixels"], "upscale": 0, "enhance": "", "retry": 0,
                "postmark_guess": 0}
elif RUN == "baseline":
    settings = {"epochs": 0, **reading}
elif RUN == "kade-baseline":
    settings = {"epochs": 0, **reading, **kade}
elif RUN == "kade":
    settings.update(**kade, **solo, epochs=5)
elif RUN == "survey-baseline":
    settings = {"epochs": 0, **reading, **survey}
elif RUN == "survey":
    settings.update(**survey, **solo, epochs=3)
elif RUN == "rescore":
    if not os.path.exists(os.path.join(ADAPTER, "adapter_config.json")):
        raise SystemExit(f"No adapter in ADAPTER = {ADAPTER!r}. Type a run's epochN folder, "
                         "e.g. os.path.join(RUNS, '<run name>', 'epoch3').")
    own = os.path.join(ADAPTER, "summary.json")   # a run folder itself, or one of its epochN folders
    trained = json.load(open(own if os.path.exists(own) else os.path.join(os.path.dirname(ADAPTER.rstrip("/")), "summary.json")))
    settings = {"epochs": 0, **reading, "adapter_dir": ADAPTER,
                "categories": trained.get("categories", "all"), "holdout_docs": trained.get("holdout_docs", "")}
elif RUN in ("kade-more", "kade-survey"):
    if not os.path.exists(os.path.join(INIT_ADAPTER, "adapter_config.json")):
        raise SystemExit(f"No adapter in INIT_ADAPTER = {INIT_ADAPTER!r}. Type a run's epochN folder, "
                         "e.g. os.path.join(RUNS, '<a kade run>', 'epoch3').")
    which = "kade_letters" if RUN == "kade-more" else "kade_letters,survey_notes"
    settings.update(categories=which, holdout_docs="kade_112", init_adapter=INIT_ADAPTER, oversample="",
                    lr=5e-5 if RUN == "kade-more" else 1e-4, epochs=2, save_every_epoch=1,
                    curve_pages=99 if RUN == "kade-more" else 10, train_score_pages=10)
elif RUN in ("dominy", "dominy-final"):
    # Only 27 Dominy training pages, so each is used 4 times per epoch (with different random
    # changes each time), and the faded ledger ink gets the contrast stretch.
    if INIT_ADAPTER and not os.path.exists(os.path.join(INIT_ADAPTER, "adapter_config.json")):
        raise SystemExit(f"No adapter in INIT_ADAPTER = {INIT_ADAPTER!r}. Set it to \"\" to start from plain CHURRO.")
    print("Starting from", INIT_ADAPTER or "plain CHURRO (a new adapter)")
    settings.update(categories="dominy_accounts", **solo)
    settings.update(oversample="dominy_accounts:4", enhance="dominy_accounts", lr=1e-4 if INIT_ADAPTER else 2e-4,
                    epochs=4 if RUN == "dominy" else FINAL_EPOCHS)
    if INIT_ADAPTER:
        settings["init_adapter"] = INIT_ADAPTER
    if RUN == "dominy-final":
        settings.update(train_on_dev=1, curve_pages=0, save_every_epoch=0, train_score_pages=0)
elif RUN in ("kade-final", "survey-final"):
    which = {"categories": "kade_letters"} if RUN == "kade-final" else survey
    settings.update(**which, oversample="", epochs=FINAL_EPOCHS, curve_pages=0, train_on_dev=1)

short = {"epochs": "ep", "lr": "lr", "lora_r": "r", "augment": "aug", "oversample": "over", "train_vision": "vis",
         "max_pixels": "px", "upscale": "up", "enhance": "enh", "retry": "re", "postmark_guess": "pm",
         "repetition_penalty": "rp"}
# Newer settings join the name only when changed, so earlier runs keep their folders.
newer = {"retry_pick": ("pick", "longer"), "cut_repeats": ("cut", ""), "no_repeat_ngram_size": ("nrng", 0),
         "retry_repetition_penalty": ("rrp", 0), "compare_adapter": ("cmp", "")}
def where_from(path):
    """A short name for an adapter folder: '<run>-epoch3', or '<run>' for a run's own adapter."""
    bits = path.rstrip("/").split("/")
    return f"{bits[-2][:24]}-{bits[-1]}" if bits[-1].startswith("epoch") else bits[-1][:32]
def tag(v):
    return str(v).replace("_accounts", "").replace("_letters", "").replace("_notes", "").replace(",", "+").replace(":", "x") or "none"
def name_of(keys):
    old = [f"{short[k]}{tag(settings[k])}" for k in keys if k in short]
    new = [f"{t}{where_from(settings[k]) if k == 'compare_adapter' else tag(settings[k])}"
           for k, (t, default) in newer.items() if k in keys and settings[k] != default]
    return "-".join(old + new)
if RUN == "full":
    run_name = "full-" + name_of(list(short) + list(newer))
elif RUN in ("kade", "survey"):
    run_name = (f"kade{SPLIT}-" if RUN == "kade" else "survey-") + name_of([k for k in list(short) + list(newer) if k != "oversample"])
elif RUN in ("kade-baseline", "survey-baseline"):
    run_name = (f"kade{SPLIT}-baseline-" if RUN == "kade-baseline" else "survey-baseline-") + name_of(READING)
elif RUN == "rescore":
    parts_of = ADAPTER.rstrip("/").split("/")
    source = parts_of[-1] if os.path.exists(own) else f"{parts_of[-2]}-{parts_of[-1]}"
    run_name = f"rescore-{source}-" + name_of(READING)
elif RUN in ("kade-more", "kade-survey"):
    run_name = f"{RUN}-from-{where_from(INIT_ADAPTER)}-" + name_of([k for k in list(short) + list(newer)
                                                                   if k not in ("oversample", "lora_r")])
elif RUN in ("dominy", "dominy-final"):
    run_name = RUN + (f"-from-{where_from(INIT_ADAPTER)}-" if INIT_ADAPTER else "-") + name_of(
        [k for k in list(short) + list(newer) if not (INIT_ADAPTER and k == "lora_r")])
elif RUN in ("kade-final", "survey-final"):
    run_name = f"{RUN}-" + name_of([k for k in list(short) + list(newer) if k != "oversample"])
else:
    run_name = RUN
run_dir = os.path.join(RUNS, run_name)
os.makedirs(run_dir, exist_ok=True)
print("This run's folder:", run_dir)


## Step 5. Train and score

The progress appears below and is saved to `train.log` in the run's folder.

**With `WHERE = "job"`:** the cell copies the data (and any adapter the run starts from) to your
bucket, only what's new, then starts the job and prints its log. The job's first 5 to 10 minutes
go on starting the machine and downloading the model. The job keeps going if the browser
disconnects or the kernel stops; you can even stop this notebook instance. Its folder is copied to
`s3://<bucket>/churro-work/runs/<run>` as it goes, and back here when it ends (or with Step 6).
Interrupting the kernel only stops *watching* it. To stop the job itself, run
`sagemaker_job.stop("<job name>")` with the name it printed, or stop it in the SageMaker console
under Training → Training jobs. A run that stopped carries on from its last checkpoint when you
start it again.

**With `WHERE = "here"`:** if the browser disconnects, it keeps going while the instance runs;
open `train.log` to see where it is. To stop it, use Kernel → Interrupt Kernel.

What to look for: `[dev before training]` and `[dev after epoch N]` (a few dev pages, for the
curve), `loss` going down, then `[dev after training]` with the full dev score per collection,
and how many pages needed a second reading.

In [ ]:
args = {**settings, "data_dir": DATA, "out_dir": run_dir}
if WHERE == "job":
    import sagemaker_job
    job_name = sagemaker_job.run_job("finetune_churro.py", args, run_dir, work=WORK, bucket=saved_setting("bucket"),
                                     run_name=run_name, instance_type=JOB_INSTANCE, max_hours=JOB_HOURS, spot=SPOT,
                                     log_name="train.log")
else:
    if not torch.cuda.is_available():
        raise SystemExit('This instance has no GPU. Set WHERE = "job" in Step 4 and run that cell again.')
    flags = " ".join(f"--{k} '{v}'" for k, v in args.items())
    log = os.path.join(run_dir, "train.log")
    python = sys.executable
    !{python} {SCRIPTS}/finetune_churro.py {flags} 2>&1 | tee -a {log}

## Step 6. The scores

With `WHERE = "job"`, this first copies the job's results from S3. So it also works after a
restart, or the next day once the job has finished: run Steps 2 and 4 with the same settings, then
this one.

CER is the share of characters wrong (lower is better). Kaggle's score is the **mean of the three
collections**, each counting for a third. Runs you've done of `"plain"` and `"baseline"` appear
next to this one, so you can see what the reading fixes and the training each add.

Kade runs also show the score **without the postmark line**, the score of **each held-out
document**, the score on 10 **training** pages (if that is far lower than the held-out score, the
model is memorising), and a table of every epoch on all the held-out pages. The best epoch's
adapter is in the run's `epochN` folder.

Then the audit's length check, per collection: a page whose reading is under 0.7 of its label's
length probably stopped early or skipped part of the page; over 1.1, it probably repeated
something or read a part twice. Fewer of both is better.

In [ ]:
import json

if WHERE == "job":
    import sagemaker_job
    sagemaker_job.fetch(run_dir, work=WORK, bucket=saved_setting("bucket"))   # the job's results, from S3
    if not os.path.exists(os.path.join(run_dir, "summary.json")):
        raise SystemExit("No results yet: the job is still running, or it failed (see train.log in the run's folder).")

summary = json.load(open(os.path.join(run_dir, "summary.json")))
table = pd.DataFrame({"this run": summary["dev_cer"]})
for other in ("plain", "baseline"):
    path = os.path.join(RUNS, other, "summary.json")
    if os.path.exists(path) and run_name != other:
        table[other] = pd.Series(json.load(open(path))["dev_cer"])
print("Dev CER (lower is better):")
print(table.round(3).to_string())
if summary.get("holdout_docs"):
    print(f"\nHeld out: {summary['holdout_docs']}; trained on: {', '.join(summary['trained_on_docs'])}")
if summary.get("dev_cer_no_postmark", {}).get("kade_letters") is not None:
    print("Kade without the postmark line:", round(summary["dev_cer_no_postmark"]["kade_letters"], 3))
if summary.get("dev_cer_by_doc"):
    print("By document:", {k: round(v, 3) for k, v in summary["dev_cer_by_doc"].items()})
if summary.get("train_pages_cer"):
    print("Training pages (the overfitting check):", {k: round(v, 3) for k, v in summary["train_pages_cer"].items()})
if summary.get("dev_cer_expert_typed_pages"):
    print("\nOn the expert-typed dev pages only:", {k: round(v, 3) for k, v in summary["dev_cer_expert_typed_pages"].items()})
print("\nLength check (pages):")
print(pd.DataFrame(summary["dev_problems"]).T.to_string())
if "best_epoch" in summary:
    print(f"\nBest epoch: {summary['best_epoch']} (dev CER {summary['best_epoch_dev_cer']:.3f}); its adapter is in "
          f"{os.path.join(run_dir, 'epoch' + str(summary['best_epoch']))}. Use it as ADAPTER for \"rescore\".")
if summary.get("init_adapter"):
    print(f"\nTrained on from the adapter {summary['init_adapter']}")
if summary.get("adapter_dir"):
    print(f"\nScored the saved adapter {summary['adapter_dir']}")
if "train_minutes" in summary:
    print(f"\nTraining took {summary['train_minutes']} min, {summary['train_images_per_epoch']} images per epoch.")

curve = json.load(open(os.path.join(run_dir, "curve.json")))
if curve["dev_cer_by_epoch"]:
    print("\nDev pages after each epoch (0 = before training):")
    print(pd.DataFrame(curve["dev_cer_by_epoch"]).round(3).to_string(index=False))
if curve["loss"]:
    import matplotlib.pyplot as plt
    loss = pd.DataFrame(curve["loss"])
    plt.figure(figsize=(6, 3)); plt.plot(loss.step, loss.loss); plt.xlabel("step"); plt.ylabel("training loss")
    plt.tight_layout(); plt.show()

### Compare two runs, page by page

A difference of 0.01 to 0.02 in the mean can be luck. This shows, for each page both runs scored,
which run did better. Trust a change only when most pages move the same way, and it holds on
both splits. Type the two run folders' names (the left-hand file browser lists them under
`churro-work/runs/`). `EPOCH` picks an epoch of a Kade run; `None` uses its final scores.


In [ ]:
OLD_RUN = "kadeA-baseline-px1003520-up1-enhnone-re1-pm1-rp1.05"   # e.g. the baseline
NEW_RUN = run_name                                                 # this run
EPOCH = None                                                       # e.g. 3, for a Kade run's epoch 3

def page_scores(run, epoch=None):
    name = f"curve_scores_epoch{epoch}.csv" if epoch is not None else "dev_scores.csv"
    return pd.read_csv(os.path.join(RUNS, run, name), keep_default_na=False)

old, new = page_scores(OLD_RUN), page_scores(NEW_RUN, EPOCH)
both = old[["page_id", "doc_id", "cer"]].merge(new[["page_id", "cer"]], on="page_id", suffixes=("_old", "_new"))
both["change"] = (both.cer_new - both.cer_old).round(3)
print(f"{len(both)} pages: {(both.change < -0.005).sum()} better, {(both.change > 0.005).sum()} worse, "
      f"{(both.change.abs() <= 0.005).sum()} about the same")
print(both.groupby("doc_id")[["cer_old", "cer_new"]].mean().round(3).to_string())
print(both.sort_values("change").round(3).to_string(index=False))


## Step 7. Look at one page

The page that scored worst first; type any dev page name to see another. Green is what the
model wrote that isn't in the label; red, struck through, is what it missed or got wrong.

In [ ]:
import difflib
import html
from IPython.display import HTML, display

def coloured_diff(reference, prediction):
    """Green: what the model wrote that isn't in the label. Red, struck through: what it missed or got wrong."""
    out = []
    for op, a0, a1, b0, b1 in difflib.SequenceMatcher(None, reference, prediction, autojunk=False).get_opcodes():
        ref, pred = html.escape(reference[a0:a1]), html.escape(prediction[b0:b1])
        if op == "equal":
            out.append(ref)
        if op in ("delete", "replace"):
            out.append(f'<span style="background:#f7c5c5;text-decoration:line-through">{ref}</span>')
        if op in ("insert", "replace"):
            out.append(f'<span style="background:#c8f7c5">{pred}</span>')
    return "".join(out).replace("\n", "<br>")

box = "font-family:monospace;line-height:1.6;border:1px solid #ccc;padding:8px"

scores = pd.read_csv(os.path.join(run_dir, "dev_scores.csv"), keep_default_na=False)
flagged = scores[scores.problem != ""].sort_values("cer", ascending=False)
if len(flagged):
    print("Pages the length check flagged:")
    print(flagged[["page_id", "cer", "length_ratio", "problem", "read_twice"]].round(3).to_string(index=False))
PAGE = scores.sort_values("cer").iloc[-1]["page_id"]   # or type a page name, e.g. "dominy_018_p002"
row = scores.set_index("page_id").loc[PAGE]
from PIL import Image
img = Image.open(os.path.join(DATA, "images", PAGE + ".jpg"))
img.thumbnail((700, 900))
display(img)
display(HTML(f"<h4>{PAGE}: CER {row['cer']:.3f}</h4><div style='{box}'>"
             f"{coloured_diff(row['reference'], row['prediction'])}</div>"))

## Step 8. Save the run to S3 (recommended)

Runs done with `WHERE = "job"` are in S3 already. The adapter is about 120 MB; the big checkpoint files are left out.

In [ ]:
MY_BUCKET = saved_setting("bucket")   # the bucket from Notebook 1; or type another, e.g. "my-bucket"

!aws s3 sync {run_dir} s3://{MY_BUCKET}/churro-work/runs/{run_name} --exclude "checkpoints/*" --quiet
print(f"Saved to s3://{MY_BUCKET}/churro-work/runs/{run_name}")

**Next:** change one setting and run Steps 4 to 8 again (for Kade, one setting at a time, with
`RUN = "kade"` on split A, then confirm the winner on split B), or go to **Notebook 3** to read the
test pages with your best run. With `WHERE = "here"`, first shut this notebook's kernel down (Kernel → Shut Down Kernel)
so the GPU is free. **When you stop for the day, stop the instance.**